# Lesson 13 — Cutting Planes

## Learning objectives

1. Derive a Gomory cut from a fractional simplex tableau {cite:p}`Gomory1958`.
2. Recognize Chvátal-Gomory, MIR, lift-and-project, and split cuts.
3. Understand cut management in modern solvers (cut pool, separation, age).
4. Use `discopt` cut callbacks.

## 1. The cutting-plane idea

If the LP relaxation $z_{LP}$ is fractional, find a linear inequality $\alpha^\top x \le \beta$ that:

- holds for every integer-feasible $x$ (valid),
- is *violated* by the current LP optimum $x_{LP}$.

Add it; resolve. Repeat. Each iteration tightens the relaxation. {cite:p}`Wolsey1998,Conforti2014` are the textbook references.

## 2. Gomory mixed-integer (GMI) cut

Take a row of the optimal LP tableau in which the basic variable $x_i$ has fractional value, with fractional part $f_i = x_i - \lfloor x_i \rfloor \in (0, 1)$. Let $f_{ij}$ be the fractional part of the row's coefficient on non-basic *integer* variable $j$, and $a_{ij}$ the row's coefficient on non-basic *continuous* variable $j$. The **Gomory mixed-integer cut** is

$$
\sum_{\substack{j \in N_I \\ f_{ij} \le f_i}} \frac{f_{ij}}{f_i} x_j \;+\; \sum_{\substack{j \in N_I \\ f_{ij} > f_i}} \frac{1 - f_{ij}}{1 - f_i} x_j \;+\; \sum_{\substack{j \in N_C \\ a_{ij} \ge 0}} \frac{a_{ij}}{f_i} x_j \;-\; \sum_{\substack{j \in N_C \\ a_{ij} < 0}} \frac{a_{ij}}{1 - f_i} x_j \;\ge\; 1.
$$

(Pure-integer Gomory cuts {cite:p}`Gomory1958` are the special case with no continuous variables.) The cut separates the current $x^{LP}$ (where every non-basic variable is at zero, so the LHS is 0) from every integer-feasible point. See {cite:p}`Wolsey1998,Conforti2014,Cornuejols2008` for the derivation.

## 3. Mixed-integer rounding (MIR)

Take a valid inequality with integer variables $x_j \in \mathbb{Z}_+$ ($j \in I$) and continuous variables $y_j \ge 0$ ($j \in C$):

$$\sum_{j \in I} a_j x_j + \sum_{j \in C} c_j y_j \le b .$$

Let $f_j = a_j - \lfloor a_j \rfloor$ and $f_b = b - \lfloor b \rfloor > 0$. The MIR inequality is

$$\sum_{j \in I} \Big(\lfloor a_j \rfloor + \frac{(f_j - f_b)^+}{1 - f_b}\Big) x_j \;+\; \frac{1}{1 - f_b}\sum_{\substack{j \in C \\ c_j < 0}} c_j\, y_j \;\le\; \lfloor b \rfloor$$

with $(z)^+ = \max(z, 0)$. Continuous terms with $c_j > 0$ are dropped (that is valid because $y_j \ge 0$). Terms with $c_j < 0$ keep a negative coefficient, scaled up by $1/(1 - f_b)$. With no continuous variables and every $f_j \le f_b$, this reduces to the Chvátal–Gomory rounding $\sum_j \lfloor a_j \rfloor x_j \le \lfloor b \rfloor$; otherwise MIR dominates it. MIR is one of the most effective cut families in modern solvers {cite:p}`Cornuejols2008`. (The GMI cut of §2 is MIR applied to a tableau row.)

## 4. Lift-and-project

For 0/1 problems: form the disjunction $x_j = 0 \vee x_j = 1$, take the convex hull of the two LP relaxations, and project onto $x$-space. Yields a polynomial-time procedure for generating cuts {cite:p}`Balas1993`.

## 5. Cut management

Modern MILP solvers add hundreds of cuts per node. They:

- **Pool** cuts that aren't currently violated.
- **Age** them out if not used in N nodes.
- Apply cuts globally (root) vs locally (single subtree).
- Use **strong cuts** (deep, dense) sparingly and **weak cuts** (sparse, cheap) often.

Cut quality is usually measured by **efficacy**: the Euclidean distance from $x_{LP}$ to the cut hyperplane, $(\alpha^\top x_{LP} - \beta)/\|\alpha\|$. Solvers also prefer cuts that are nearly orthogonal to the cuts already chosen, so that a round does not add near-duplicates.

### Cuts inside `discopt`

- By default a pure MILP is handed to **HiGHS**, which runs its own cut loop; `discopt` then checks the certificate HiGHS returns (see `r.algorithm_route`).
- With `milp_backend="native"`, `discopt`'s own MILP tree runs instead. Before branching it runs a root cut loop of up to 5 rounds of cover, clique, Gomory mixed-integer and MIR cuts, and `m.solve(milp_backend="native", milp_cuts=False)` switches that loop off. `r.solver_stats` then counts the cuts by family (`"cuts/cover_clique"`, `"cuts/gomory"`, `"cuts/mir"`), and `r.root_bound` is the bound after the cut loop. Comparing the two runs measures what the cuts buy (see the cell below and Exercise 4).
- With `nlp_solver="simplex"`, a second engine runs: `discopt`'s monolithic Rust branch-and-cut. Its root cut loop separates Gomory mixed-integer cuts from the simplex tableau, row MIR cuts, and knapsack cover cuts (`crates/discopt-core/src/lp/gomory.rs`, `mir.rs`, `cover.rs`). It keeps a few strong cuts per round, chosen by efficacy plus orthogonality (`cut_select.rs`), for up to 50 rounds while the bound is still improving. Cover cuts are also separated at fractional nodes into a global cut pool.
- You can add your *own* cuts with `m.solve(cut_callback=...)` (Exercise 5). Every cut you return is checked against the feasible points the solve has already verified; a cut that removes one raises `CutValidationError` rather than being silently trusted.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do

# Toy MILP: max x1 + x2 s.t. 2x1 + x2 <= 5, x1 + 3x2 <= 6, x1,x2 in {0,1,...,5}.
# On a model this small every engine finishes at the root, so the useful
# measurement is the *bound*: solve the LP
# relaxation (continuous variables) with and without a hand-derived
# Chvatal-Gomory cut and compare both to the integer optimum.
def build(extra_cut: bool, relax: bool):
    m = do.Model("cut_demo")
    if relax:
        x = m.continuous("x", shape=(2,), lb=0, ub=5)
    else:
        x = m.integer("x", shape=(2,), lb=0, ub=5)
    m.subject_to(2*x[0] + x[1] <= 5)
    m.subject_to(x[0] + 3*x[1] <= 6)
    if extra_cut:
        # CG cut, u = (1/3, 1/3): x1 + (4/3) x2 <= 11/3  ->  x1 + x2 <= 3
        m.subject_to(x[0] + x[1] <= 3)
    m.maximize(x[0] + x[1])
    return m

z_ip = build(False, relax=False).solve().objective
for label, ec in [("LP relaxation", False), ("LP + CG cut", True)]:
    z_lp = build(ec, relax=True).solve().objective
    print(f"{label:14s}  bound={z_lp:.3f}   integer optimum={z_ip:.0f}   gap={z_lp - z_ip:.3f}")


In [ ]:
# A larger instance, where the root cut loop matters: a 5-row, 40-binary
# multi-dimensional knapsack, solved on discopt's own MILP tree with and
# without its root cuts. The LP relaxation bound comes from relaxing x to [0, 1].
rng = np.random.default_rng(0)
A = rng.integers(5, 50, size=(5, 40))
v = rng.integers(5, 50, size=40)

def mdkp(relax=False):
    m = do.Model("mdkp")
    x = m.continuous("x", shape=(40,), lb=0, ub=1) if relax else m.binary("x", shape=(40,))
    m.subject_to(A @ x <= A.sum(axis=1) // 2)
    m.maximize(v @ x)
    return m

z_lp = mdkp(relax=True).solve().objective
for cuts in (True, False):
    r = mdkp().solve(milp_backend="native", milp_cuts=cuts)
    n_cuts = sum(int(r.solver_stats.get(k, 0)) for k in ("cuts/cover_clique", "cuts/gomory", "cuts/mir"))
    closed = (z_lp - r.root_bound) / (z_lp - r.objective)
    print(f"milp_cuts={cuts!s:5s}  cuts={n_cuts:3d}  root bound={r.root_bound:.2f}  "
          f"root gap closed={closed:5.1%}  nodes={r.node_count:4d}  z*={r.objective:.0f}")
print(f"LP relaxation bound = {z_lp:.2f}")


## References
{cite:p}`Gomory1958` (original), {cite:p}`Balas1993` (lift-and-project),
{cite:p}`Cornuejols2008` (modern survey),
{cite:p}`Wolsey1998,Conforti2014` (textbooks).